In [1]:
from langchain_classic.document_loaders import TextLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser



c:\Users\mendh\Langchain-Langgraph\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
loader = DirectoryLoader(r'C:\Users\mendh\Langchain-Langgraph\0-Dataingestion-parsing\data\text_files', glob='**/*.txt', show_progress=True,loader_cls=TextLoader)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)
print(f"Number of documents: {len(docs)}")

100%|██████████| 4/4 [00:00<00:00, 1837.79it/s]

Number of documents: 28


In [3]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriver = vectorstore.as_retriever(search_kwargs={"k": 5})
retriver

C:\Users\mendh\AppData\Local\Temp\ipykernel_12504\2168686724.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002B1162C8AD0>, search_kwargs={'k': 5})

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
llm = init_chat_model(model_provider="groq", model="llama-3.1-8b-instant")
llm.invoke("Hello, how are you?")

AIMessage(content="I'm functioning properly, thank you for asking. I'm a large language model, so I don't have emotions or feelings like humans do, but I'm here to help answer any questions or provide information you need. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 41, 'total_tokens': 93, 'completion_time': 0.067491206, 'completion_tokens_details': None, 'prompt_time': 0.001991573, 'prompt_tokens_details': None, 'queue_time': 0.045583527, 'total_time': 0.069482779}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d485f-ac3c-76d2-8457-a0f1eddb20b9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 52, 'total_tokens': 93})

In [5]:
query  = PromptTemplate.from_template("""you are a helpful assistant. Decompose the following complex question into 2-4 smaller sub-questions for better document retrival.
original query:{query}:

sub-questions:""")

In [6]:
query_expansion_chain = query | llm | StrOutputParser()

In [7]:
print(query_expansion_chain.invoke({"query": "Langchain memory"}))

Here's a decomposition of the original query into smaller sub-questions for better document retrieval:

1. **What is Langchain memory?**
   - This sub-question aims to understand the fundamental concept of Langchain memory and its definition.

2. **How does Langchain memory store and manage information?**
   - This sub-question focuses on the storage and management aspects of Langchain memory, including possible data structures and retrieval mechanisms.

3. **What are the key features and benefits of Langchain memory?**
   - This sub-question seeks to identify the primary advantages and characteristics of Langchain memory, such as scalability, efficiency, or security.

4. **How does Langchain memory support AI applications and use cases?**
   - This sub-question explores the role of Langchain memory in AI development, including its potential applications, integration with other AI systems, and impact on various industries.

These sub-questions provide a clear direction for document ret